# `parse_word` + `apply_changes` — pattern extract / modify / rebuild

Module testé : [`src/docpipeline/parsing/word/parse_word.py`](../src/docpipeline/parsing/word/parse_word.py).

## Le pattern transversal du projet

Pour traduction, conversion, redaction — partout où on doit modifier du texte SANS perdre les styles :

```
extract  : parse_word(docx)             → spans avec styles + span_id stable
modify   : on change SEULEMENT le text de chaque span (translate, redact, …)
rebuild  : apply_changes(docx_in, {span_id: new_text}, docx_out)
           → reconstruit le .docx avec nouveaux textes mais styles intacts
```

Le `span_id` (format `w_<para_idx>_<run_idx>`) est la **clé stable** qui fait le pont entre extract et rebuild — robuste aux textes dupliqués.

## Sortie de `parse_word(docx)`

| Sortie | Contenu |
|---|---|
| `paragraph_df` | 1 ligne = 1 paragraphe (style, alignement, indents, list_level, heading_level, ...) |
| `span_df`      | 1 ligne = 1 run (span_id, text, font_name, font_size_pt, bold, italic, underline, color, highlight, subscript, superscript, all_caps, small_caps, hyperlink, is_insertion/deletion, ...) |
| `image_df`     | 1 ligne = 1 image embarquée |
| `table_df`     | 1 ligne = 1 table (n_rows, n_cols, style, alignment) |
| `doc_summary`  | JSON synthèse : metadata, source_tool, has_toc, has_track_changes, has_comments, has_footnotes, style_counts, ... |

Démo ci-dessous : extraction sur `tests/fixtures/contrat_assurance.docx`, puis modification de 3 spans (uppercase) et rebuild — on vérifie que les styles sont intacts. Aucun LLM (cf. `CLAUDE.md`).

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import json
from pathlib import Path
from docpipeline.parsing.word.parse_word import parse_word, apply_changes

DOCX_IN  = Path('../tests/fixtures/contrat_assurance.docx')
DOCX_OUT = Path('../tests/fixtures/out_modified_js.docx')

# 1) EXTRACT
result = parse_word(DOCX_IN)
print('=== doc_summary ===')
print(json.dumps(result['doc_summary'], indent=2, ensure_ascii=False, default=str))
print()
print('=== span_df (5 premiers) ===')
print(result['span_df'].head(5)[['span_id', 'text', 'font_name', 'font_size_pt', 'bold', 'italic', 'color']].to_string(index=False))

# 2) MODIFY : uppercase les 3 premiers spans non vides
changes = {row['span_id']: row['text'].upper()
           for _, row in result['span_df'].head(3).iterrows()
           if row['text'].strip()}
print()
print(f'=== modifications ({len(changes)} spans -> uppercase) ===')
for sid, txt in changes.items():
    print(f'  {sid} : {txt[:70]}')

# 3) REBUILD
apply_changes(DOCX_IN, changes, DOCX_OUT)
result2 = parse_word(DOCX_OUT)
print()
print('=== span_df apres rebuild (5 premiers) ===')
print(result2['span_df'].head(5)[['span_id', 'text', 'font_name', 'font_size_pt', 'bold', 'italic', 'color']].to_string(index=False))

# 4) VERIF : styles preserves apres modification du texte
print()
print('=== Styles preserves apres rebuild ? ===')
before = result['span_df']
after  = result2['span_df']
checks = ['font_name', 'font_size_pt', 'bold', 'italic', 'underline', 'color', 'highlight']
for col in checks:
    same = (before[col].fillna('').astype(str) == after[col].fillna('').astype(str)).all()
    print(f'  {col:15s} : {"OK" if same else "DIFF"}')

# Cleanup
if DOCX_OUT.exists():
    DOCX_OUT.unlink()
print('\n(out_modified_js.docx supprime apres verif)')

=== doc_summary ===
{
  "doc_hash": "0134552717eb2b56e4ae7b799c060f7e70ed838968e293f3c9ea6c5faa9eb5c2",
  "file_size_bytes": 37367,
  "n_paragraphs": 10,
  "n_spans": 11,
  "n_images": 0,
  "n_tables": 1,
  "n_sections": 1,
  "n_headings": 6,
  "n_list_items": 0,
  "total_char_count": 689,
  "source_tool": "Microsoft Word",
  "metadata": {
    "title": "",
    "author": "Faseya Test",
    "last_modified_by": "",
    "subject": "",
    "keywords": "",
    "category": "",
    "comments": "generated by python-docx",
    "revision": 1,
    "created": "2013-12-23T23:15:00+00:00",
    "modified": "2013-12-23T23:15:00+00:00",
    "last_printed": null,
    "version": "",
    "content_status": "",
    "language": ""
  },
  "has_toc": true,
  "has_track_changes": false,
  "has_comments": false,
  "has_footnotes": false,
  "has_hyperlinks": false,
  "style_counts": {
    "Heading 1": 4,
    "Normal": 4,
    "Title": 1,
    "Heading 2": 1
  },
  "recommended_strategy": "use_native_extraction",
  "


=== span_df apres rebuild (5 premiers) ===
span_id                                                                                                                                                                                                text font_name font_size_pt  bold  italic color
  w_0_0                                                                                                                                                          CONTRAT D'ASSURANCE — CONDITIONS GÉNÉRALES                   None False   False   NaN
  w_1_0                                                                                                                                                                                 1. OBJET DU CONTRAT                   None False   False   NaN
  w_2_0 LE PRÉSENT CONTRAT (IA) COUVRE LES ACCIDENTS INDIVIDUELS SURVENUS DANS LE CADRE DE L'ACTIVITÉ PROFESSIONNELLE DE L'ASSURÉ. LA GARANTIE BI (BUSINESS INTERRUPTION) EST APPLICABLE SELON L'ARTICLE 5.            